<a href="https://colab.research.google.com/github/pratheerth/oreilly_slm/blob/main/O'Reilly_SLM_Part1_Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small Language Models - Part 1 - Fundamentals

In [ ]:
# install required libraries
%pip install transformers accelerate torch sentencepiece bitsandbytes --quiet
print("Libraries installed!")

Libraries installed!


In [2]:
# Log in to Hugging Face
# from huggingface_hub import notebook_login
# notebook_login()
from huggingface_hub import interpreter_login
interpreter_login()

f:\GitHub\oreilly_slm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
# define model ID
model_id_llama = "meta-llama/Llama-3.2-1B-Instruct"

# load tokenizer
tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama)

# load model
model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    torch_dtype=torch.bfloat16,
    # load_in_4bit=True,
    device_map="auto"
)
print("Llama 3.2 1B Instruct loaded")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 1474.90it/s, Materializing param=model.norm.weight]                              


Llama 3.2 1B Instruct loaded


In [ ]:
# llama verification
print("Running quick verification...")
_prompt = "Hi! How are you today?" # define a simple input string
# convert the prompt text to tokens, make it a pytorch tensor,
# and send it to the same device the model is on (the GPU)
_inputs = tokenizer_llama(_prompt, return_tensors="pt").to(model_llama.device)

# ask the model to generate a response based on the input
# max_new_tokens limits the output length to 5 tokens for quick check
_outputs = model_llama.generate(**_inputs, max_new_tokens=5)

# decode the numeric output tokens back into readable text
# outputs[0] ges the first sequence in the batch
print("Llama Verification Ok:", tokenizer_llama.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

In [ ]:
!nvidia-smi

In [5]:
# --- Load Gemma 2 2B ---
model_id_gemma = "google/gemma-2-2b-it"
tokenizer_gemma = AutoTokenizer.from_pretrained(model_id_gemma)
model_gemma = AutoModelForCausalLM.from_pretrained(
    model_id_gemma,
    torch_dtype=torch.bfloat16,
    # load_in_4bit=True,
    device_map="auto"
)
print(f"Gemma 2 2B loaded")

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 1319.64it/s, Materializing param=model.norm.weight]                                


Gemma 2 2B loaded


In [ ]:
# gemini verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_gemma(_prompt, return_tensors="pt").to(model_gemma.device)
_outputs = model_gemma.generate(**_inputs, max_new_tokens=5)

print("Gemma Verification Ok:", tokenizer_gemma.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

In [ ]:
!nvidia-smi

In [6]:
# --- Load Qwen 2.5 0.5B ---
model_id_qwen = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id_qwen)
model_qwen = AutoModelForCausalLM.from_pretrained(
    model_id_qwen,
    torch_dtype=torch.bfloat16,
    # load_in_4bit=True,
    device_map="auto"
)
print(f"Qwen 2.5 0.5B loaded")

# qwen verification
print("Running quick verification...")
_prompt = "Hi! How are you today?"
_inputs = tokenizer_qwen(_prompt, return_tensors="pt").to(model_qwen.device)
_outputs = model_qwen.generate(**_inputs, max_new_tokens=5)
print("Qwen Verification Ok:", tokenizer_qwen.decode(_outputs[0]))

# cleaning up temporary variables
del _prompt, _inputs, _outputs

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1444.29it/s, Materializing param=model.norm.weight]                              


Qwen 2.5 0.5B loaded
Running quick verification...
Qwen Verification Ok: Hi! How are you today? I'm sorry to say


In [7]:
!nvidia-smi

Tue Mar  3 20:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   53C    P8              9W /  150W |       0MiB /  24463MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
import time # Import the time library

# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [ ]:
prompt = "Describe the year-round climate in London, England"

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt, max_new_tokens=500)

In [ ]:
prompt_2 = """Scenario:
Four colored cups – Red, Blue, Green, and Yellow – are arranged in a circle.
One cup has a hidden star.

Facts:

The cup with the star is not Red.
The cup with the star is directly next to the Blue cup.
The Green cup is directly between the Yellow cup and the Red cup (when going around the circle in one direction).
Question:
Which color cup has the star? Just give the correct answer"""

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_2, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_2, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_2, max_new_tokens=500)

In [ ]:
prompt_3 = """Write a simple Python function called `calculate_area`
that accepts the length and width of a rectangle and returns the area."""

In [ ]:
# Generate with Llama 3.2 1B
llama_output, llama_speed = generate_text(model_llama, tokenizer_llama, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Gemma 2 2B
gemma_output, gemma_speed = generate_text(model_gemma, tokenizer_gemma, prompt_3, max_new_tokens=500)

In [ ]:
# Generate with Qwen 2.5 0.5B
qwen_output, qwen_speed = generate_text(model_qwen, tokenizer_qwen, prompt_3, max_new_tokens=500)

In [9]:
prompt = "Write a short story about a time-traveling cat"

In [10]:
import time
# Define a function to handle text generation and timing
def generate_text(model, tokenizer, prompt, max_new_tokens, temperature):

    start_time = time.time() # Record the time before generation starts

    # Prepare the input: tokenize, convert to PyTorch tensors, move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text using the model
    outputs = model.generate(
        **inputs,                     # Pass the tokenized inputs
        max_new_tokens=max_new_tokens, # Set maximum number of new tokens to generate
        temperature=temperature,      # Control the randomness (creativity vs focus)
        do_sample=True,               # Enable sampling (needed for temperature)
        #pad_token_id=tokenizer.eos_token_id # Prevent warnings about padding token
    )
    end_time = time.time() # Record the time after generation finishes

    # Process the output
    output_ids = outputs[0] # Get the full sequence of token IDs
    input_token_len = inputs.input_ids.shape[1] # Find length of original input tokens
    generated_ids = output_ids[input_token_len:] # Isolate the newly generated token IDs
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True) # Convert generated IDs to text
    num_generated_tokens = len(generated_ids) # Count how many tokens were generated

    # Calculate and display performance
    duration = end_time - start_time
    tokens_per_sec = num_generated_tokens / duration
    print(f"Generated {num_generated_tokens} tokens in {duration:.2f} seconds ({tokens_per_sec:.2f} tokens/sec)")

    # Print the final generated text
    print("Output:")
    print(generated_text)

    # Return the text and speed for potential later use
    return generated_text, tokens_per_sec

In [11]:
# low temperature
_ = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=0.2)

Generated 500 tokens in 55.46 seconds (9.02 tokens/sec)
Output:
 named Whiskers.

Professor Elm, a renowned time-traveling scientist, had a secret. He wasn't alone in his pursuit of the impossible. Whiskers, a ginger tabby with a penchant for adventure, was his constant companion. 

Whiskers wasn't your average cat. He possessed a unique ability: he could travel through time. It wasn't a conscious choice, but a consequence of Professor Elm's experiments. Whiskers' fur would glow with an ethereal light, and he'd disappear in a puff of smoke, only to reappear in a different era.

One day, Professor Elm, with Whiskers tucked safely in his pocket, decided to visit the Roman Empire. The air was thick with the scent of spices and the cacophony of a bustling marketplace. Whiskers, however, was more interested in the pigeons than the crowds. He found himself perched atop a crumbling archway, watching the pigeons strut and peck.

Suddenly, a commotion erupted. A group of men, dressed in togas, 

In [12]:
# high temperature
_ = generate_text(model_gemma, tokenizer_gemma, prompt, max_new_tokens=500, temperature=1.0)

Generated 500 tokens in 58.54 seconds (8.54 tokens/sec)
Output:
, whose fur is covered in tiny quills.

*whispers*  He's called Chronos.

He'd been traveling for years, a silent observer of history, his soft paws padded against the cobblestones of ancient cities and his luminous green eyes taking in the magnificent details. As he purred, his tiny quills, a peculiar byproduct of time travel, would softly tingle, emitting faint bursts of light.  Chronos loved the feeling of being a silent witness to history, each new adventure leaving his fur tingling with the echoes of the past.

One day, he landed in a bustling medieval market. It wasn't what he'd expected. A pungent aroma assaulted his sensitive nose - a symphony of spices, roasting meat, and...wine? Chronos sniffed the air greedily, his whiskers twitching. The scent was intoxicating. 

Suddenly, a commotion erupted. A young girl, no older than ten, stumbled upon a street vendor's stall filled with exotic spices. She was trying to pic